# Notebook 03 — Feature Engineering

**What this notebook builds:**
1. **TF-IDF features** — sparse text representation (unigrams + bigrams, 30k vocab)
2. **Metadata features** — structured signals from `submitted_via`, `tags`, `narrative_len`, `state` region
3. **Combined feature matrix** — TF-IDF + metadata concatenated as a sparse matrix
4. **Top terms per class** — interpretability: what words drive each product category

**Why TF-IDF before BERT?**  
TF-IDF + XGBoost is our **baseline** and often our strongest classical model. It trains in seconds, is interpretable, and sets the benchmark that BERT needs to beat. We fit the vectorizer **on train only** — transforming val/test separately — to prevent data leakage.

**Output files saved to Drive:**
- `tfidf_vectorizer.joblib` — fitted vectorizer (reused at inference time)
- `meta_encoder.joblib` — fitted metadata encoder
- `X_train.npz`, `X_val.npz`, `X_test.npz` — combined sparse feature matrices
- `y_train.npy`, `y_val.npy`, `y_test.npy` — label arrays

## Cell 1 — Mount Drive & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR  = '/content/drive/MyDrive/nlp_project'
DATA_DIR     = os.path.join(PROJECT_DIR, 'data')
FEATURE_DIR  = os.path.join(PROJECT_DIR, 'features')
OUTPUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(FEATURE_DIR, exist_ok=True)

TEXT_COL  = 'narrative_clean'
LABEL_COL = 'label'

print('Ready.')

## Cell 2 — Load Train / Val / Test Splits

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# Load label mapping for display
with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping   = json.load(f)
    id2label  = {int(k): v for k, v in mapping['id2label'].items()}
    label2id  = mapping['label2id']

print(f'Train : {len(df_train):,}')
print(f'Val   : {len(df_val):,}')
print(f'Test  : {len(df_test):,}')
print(f'Columns: {df_train.columns.tolist()}')

## Cell 3 — TF-IDF Vectoriser

**Key choices:**
- `ngram_range=(1,2)` — unigrams + bigrams. Bigrams catch phrases like "credit card" vs just "credit"
- `stop_words='english'` — removes 'the', 'and', 'to', 'my', etc. so only meaningful terms rank in the vocabulary
- `max_features=30_000` — vocabulary cap after stopword removal
- `min_df=5` — ignore terms appearing in fewer than 5 documents (noise / typos)
- `sublinear_tf=True` — applies log(1+tf) instead of raw tf, dampens very frequent terms
- `token_pattern` — ignores pure-number tokens like phone numbers

**Fit ONLY on train. Then transform val and test separately** — prevents data leakage.

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range  = (1, 2),
    max_features = 30_000,
    min_df       = 5,
    sublinear_tf = True,
    stop_words   = 'english',              # removes 'the','and','to','my','they', etc.
    analyzer     = 'word',
    token_pattern= r'\b[a-zA-Z][a-zA-Z0-9]+\b',
)

print('Fitting TF-IDF on train ...')
X_tfidf_train = tfidf.fit_transform(df_train[TEXT_COL].fillna(''))

print('Transforming val and test ...')
X_tfidf_val  = tfidf.transform(df_val[TEXT_COL].fillna(''))
X_tfidf_test = tfidf.transform(df_test[TEXT_COL].fillna(''))

print(f'\nVocabulary size   : {len(tfidf.vocabulary_):,}')
print(f'Train TF-IDF shape: {X_tfidf_train.shape}  (sparse)')
print(f'Val   TF-IDF shape: {X_tfidf_val.shape}')
print(f'Test  TF-IDF shape: {X_tfidf_test.shape}')
print(f'Sparsity (train)  : {1 - X_tfidf_train.nnz / (X_tfidf_train.shape[0]*X_tfidf_train.shape[1]):.3%}')

tfidf_path = os.path.join(FEATURE_DIR, 'tfidf_vectorizer.joblib')
joblib.dump(tfidf, tfidf_path)
print(f'\nSaved: {tfidf_path}')

## Cell 4 — Top TF-IDF Terms Per Class

For each class, we compute the mean TF-IDF score across all training documents in that class.
The highest-scoring terms are the most distinctive vocabulary for that product category.

This is a powerful sanity check: if "mortgage" ranks highest for the Mortgage class and
"debt" ranks highest for Debt Collection, the features are working correctly.

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
y_train       = df_train[LABEL_COL].values

print('Top 10 TF-IDF terms per class:\n')
for label_id in sorted(id2label):
    mask      = (y_train == label_id)
    mean_tfidf = np.asarray(X_tfidf_train[mask].mean(axis=0)).flatten()
    top_idx   = mean_tfidf.argsort()[-10:][::-1]
    top_terms = feature_names[top_idx]
    print(f'{id2label[label_id]}')
    print(f'  {list(top_terms)}')
    print()

## Cell 5 — Metadata Features

Beyond the text, we have structured fields that carry real signal:

| Feature | Type | Encoding | Why useful |
|---|---|---|---|
| `submitted_via` | Categorical | One-hot | Channel (Web vs Phone vs Fax) correlates with product type |
| `tags` | Multi-value string | Binary flags | "Older American" and "Servicemember" tags correlate with specific complaint types |
| `narrative_len` | Numeric | StandardScaler | Longer complaints correlate with mortgage/loan disputes |
| `state_region` | Categorical | One-hot | Geographic patterns in complaint types |

We skip `company` (thousands of unique values) and raw `state` (51 values would dominate one-hot).  
Instead we map states to 4 US census regions.

In [ ]:
REGION_MAP = {
    'CT':'Northeast','ME':'Northeast','MA':'Northeast','NH':'Northeast',
    'RI':'Northeast','VT':'Northeast','NJ':'Northeast','NY':'Northeast','PA':'Northeast',
    'IN':'Midwest','IL':'Midwest','MI':'Midwest','OH':'Midwest','WI':'Midwest',
    'IA':'Midwest','KS':'Midwest','MN':'Midwest','MO':'Midwest','NE':'Midwest',
    'ND':'Midwest','SD':'Midwest',
    'DE':'South','FL':'South','GA':'South','MD':'South','NC':'South','SC':'South',
    'VA':'South','DC':'South','WV':'South','AL':'South','KY':'South','MS':'South',
    'TN':'South','AR':'South','LA':'South','OK':'South','TX':'South',
    'AZ':'West','CO':'West','ID':'West','MT':'West','NV':'West','NM':'West',
    'UT':'West','WY':'West','AK':'West','CA':'West','HI':'West','OR':'West','WA':'West',
}

def build_metadata(df_in: pd.DataFrame) -> pd.DataFrame:
    # Fully vectorized — no iterrows() — handles 140k rows instantly
    tags = df_in['tags'].fillna('').astype(str)
    return pd.DataFrame({
        'submitted_via'     : df_in['submitted_via'].fillna('Unknown').astype(str),
        'state_region'      : df_in['state'].fillna('').astype(str).map(REGION_MAP).fillna('Unknown'),
        'is_older_american' : tags.str.contains('Older American', na=False).astype(int),
        'is_servicemember'  : tags.str.contains('Servicemember',  na=False).astype(int),
        'narrative_len'     : df_in['narrative_len'].fillna(0).astype(float),
    })

print('Building metadata frames ...')
meta_train = build_metadata(df_train)
meta_val   = build_metadata(df_val)
meta_test  = build_metadata(df_test)
print(f'Metadata shape: {meta_train.shape}')
print(meta_train.head(3).to_string())

In [ ]:
# One-hot encode categoricals, scale numeric — fit on train only
CAT_COLS = ['submitted_via', 'state_region']
NUM_COLS = ['is_older_american', 'is_servicemember', 'narrative_len']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
scaler = StandardScaler()

# Fit on train
cat_train = ohe.fit_transform(meta_train[CAT_COLS])
num_train = scaler.fit_transform(meta_train[NUM_COLS])

# Transform val and test
cat_val  = ohe.transform(meta_val[CAT_COLS])
num_val  = scaler.transform(meta_val[NUM_COLS])
cat_test = ohe.transform(meta_test[CAT_COLS])
num_test = scaler.transform(meta_test[NUM_COLS])

# Stack numeric as sparse too
num_train_sp = sp.csr_matrix(num_train)
num_val_sp   = sp.csr_matrix(num_val)
num_test_sp  = sp.csr_matrix(num_test)

print(f'OHE categories : {ohe.get_feature_names_out().tolist()}')
print(f'Metadata train shape (OHE+numeric): {sp.hstack([cat_train, num_train_sp]).shape}')

# Save encoders
joblib.dump({'ohe': ohe, 'scaler': scaler}, os.path.join(FEATURE_DIR, 'meta_encoder.joblib'))
print(f'Saved meta_encoder.joblib')

## Cell 6 — Combine TF-IDF + Metadata → Final Feature Matrix

We `hstack` (horizontally stack) the sparse TF-IDF matrix with the sparse metadata matrix.
The result is a single sparse matrix per split that XGBoost will consume directly.

Shape will be: `(n_rows, 30_000 tfidf + n_meta_features)`

In [ ]:
X_train = sp.hstack([X_tfidf_train, cat_train, num_train_sp], format='csr')
X_val   = sp.hstack([X_tfidf_val,   cat_val,   num_val_sp],   format='csr')
X_test  = sp.hstack([X_tfidf_test,  cat_test,  num_test_sp],  format='csr')

y_train = df_train[LABEL_COL].values
y_val   = df_val[LABEL_COL].values
y_test  = df_test[LABEL_COL].values

print(f'X_train : {X_train.shape}  nnz={X_train.nnz:,}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}  classes={np.unique(y_train)}')

# Save sparse matrices
for name, mat in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test)]:
    path = os.path.join(FEATURE_DIR, f'{name}.npz')
    sp.save_npz(path, mat)
    print(f'Saved {name}.npz  ({os.path.getsize(path)/1e6:.1f} MB)')

# Save label arrays
for name, arr in [('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    path = os.path.join(FEATURE_DIR, f'{name}.npy')
    np.save(path, arr)
    print(f'Saved {name}.npy')

## Cell 7 — Summary (paste this output to Claude)

In [ ]:
print('='*60)
print('  NOTEBOOK 03 SUMMARY — paste this output to Claude')
print('='*60)
print(f'TF-IDF vocabulary size     : {len(tfidf.vocabulary_):,}')
print(f'TF-IDF ngram range         : (1, 2)')
print(f'OHE metadata features      : {ohe.get_feature_names_out().tolist()}')
print(f'Numeric metadata features  : {NUM_COLS}')
print(f'Total features per sample  : {X_train.shape[1]:,}')
print(f'X_train shape              : {X_train.shape}')
print(f'X_val shape                : {X_val.shape}')
print(f'X_test shape               : {X_test.shape}')
print(f'Sparsity (train)           : {1 - X_train.nnz/(X_train.shape[0]*X_train.shape[1]):.4%}')
print()
print('Feature matrix file sizes (MB):')
for name in ['X_train.npz','X_val.npz','X_test.npz']:
    path = os.path.join(FEATURE_DIR, name)
    print(f'  {name:<15} {os.path.getsize(path)/1e6:.1f} MB')
print('='*60)